In [19]:
import os
from google import genai
import json
from IPython.display import Markdown, display, update_display
from google.genai import types 
import base64
from dotenv import load_dotenv
from WebScraper import fetch_website_links, fetch_website_content

In [2]:
# Load environment variables
env_path = r"C:\Users\Cem\Desktop\projeler\YapayZekaUygulamalari\.env"
load_dotenv(env_path)

True

In [4]:
api_key = os.getenv("GEMINI_API_KEY")

In [5]:
if not api_key:
     print("No Api Key Has been Found");
else:
     print("Api key has been found and looks good so far");

Api key has been found and looks good so far


{'model': 'gemini-2.0-flash',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [22]:
links = fetch_website_links("https://www.trendyol.com/")
links

['https://www.trendyol.com/Hesabim/IndirimKuponlari',
 'https://partner.trendyol.com/onboarding/satici-formu',
 'https://www.trendyol.com/s/meet-us#whoweare',
 'https://www.trendyol.com/iphone-17-cep-telefonu-x-c103498-a292-v10632549',
 'https://www.trendyol.com/iphone-17-air-cep-telefonu-x-c103498-a292-v10632550',
 'https://www.trendyol.com/iphone-17-pro-cep-telefonu-x-c103498-a292-v10632551',
 'https://www.trendyol.com/iphone-17-pro-max-cep-telefonu-x-c103498-a292-v10632552',
 'https://www.trendyol.com/iphone-16-cep-telefonu-x-c103498-a292-v10577547',
 'https://www.trendyol.com/iphone-16-plus-cep-telefonu-x-c103498-a292-v10591876',
 'https://www.trendyol.com/iphone-16-pro-cep-telefonu-x-c103498-a292-v10577551',
 'https://www.trendyol.com/apple/iphone-16-pro-max-256gb-siyah-titanyum-p-857296077',
 'https://www.trendyol.com/apple/iphone-16e-128gb-siyah-p-900754126',
 'https://www.trendyol.com/iphone-13-cep-telefonu-x-c103498-a292-v613184',
 'https://www.trendyol.com/iphone-13-mini-cep-

In [23]:
link_system_prompt = """
     you are provided with a list of links found in a online shopping web page.
     Your mission is to retrieve the products by categorizing them for instance you will look at the 
     links of the products and you will categorize those product such as for the https://www.trendyol.com/iphone-15-pro-max-cep-telefonu-x-c103498-a292-v1218522 link
     you will categorize it as technology and so on you will JSON as in this example
     
     [
          {"category": "technology", "url": "https://www.trendyol.com/iphone-15-pro-max-cep-telefonu-x-c103498-a292-v1218522 link"},
          {"category": "cosmetic", "url": "https://www.trendyol.com/the-ceel/arindirici-ve-aydinlatici-yuz-peelingi-beluga-havyari-ozlu-coenzyme-q10-100-ml-p-780950160?merchantId=791156&boutiqueId=61"}
     ]
"""

In [24]:
def get_links_user_prompt(url):
     user_prompt = f"""
          Here is the list of links on e-shoping website {url} -
          please decide the category of each product by looking at their links
          respond with full HTTPS url in JSON format
          Just include the product links nothing else like email, Service, Privacy or any other links.
          
          Links (some might be relative links)
          
     """
     
     links = fetch_website_links(url);
     user_prompt += "\n".join(links);
     return user_prompt;

In [25]:
print(get_links_user_prompt("https://www.trendyol.com/"));


          Here is the list of links on e-shoping website https://www.trendyol.com/ -
          please decide the category of each product by looking at their links
          respond with full HTTPS url in JSON format
          Just include the product links nothing else like email, Service, Privacy or any other links.
          
          Links (some might be relative links)
          
     https://www.trendyol.com/Hesabim/IndirimKuponlari
https://partner.trendyol.com/onboarding/satici-formu
https://www.trendyol.com/s/meet-us#whoweare
https://www.trendyol.com/iphone-17-cep-telefonu-x-c103498-a292-v10632549
https://www.trendyol.com/iphone-17-air-cep-telefonu-x-c103498-a292-v10632550
https://www.trendyol.com/iphone-17-pro-cep-telefonu-x-c103498-a292-v10632551
https://www.trendyol.com/iphone-17-pro-max-cep-telefonu-x-c103498-a292-v10632552
https://www.trendyol.com/iphone-16-cep-telefonu-x-c103498-a292-v10577547
https://www.trendyol.com/iphone-16-plus-cep-telefonu-x-c103498-a292-v10591876

In [26]:
def select_relative_links(url):
     model_name = "gemini-2.0-flash"
     client = genai.Client(
          api_key=os.getenv("GEMINI_API_KEY")
     )
     
     # Get user prompt with links
     user_prompt = get_links_user_prompt(url)
     
     # CORRECT WAY: Combine system and user prompts into a single prompt
     combined_prompt = f"{link_system_prompt}\n\n{user_prompt}"
     
     # Create contents with the combined prompt
     contents = [
         types.Content(
             role="user",  # Only "user" and "model" roles exist in Gemini
             parts=[
                 types.Part.from_text(text=combined_prompt),
             ],
         ),
     ]
     
     # Generate content
     try:
         response = client.models.generate_content(
             model=model_name,
             contents=contents,
         )
         return response.text
     except Exception as e:
         return f"Error generating content: {str(e)}"

In [28]:
import os
import json
from datetime import datetime

def save_result_to_json(result, url, folder_name="results"):
    """
    Save the result to a JSON file in the specified folder
    """
    # Create results folder if it doesn't exist
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    domain = url.replace("https://", "").replace("http://", "").replace("/", "_")
    filename = f"{domain}_{timestamp}.json"
    filepath = os.path.join(folder_name, filename)
    
    # Prepare data to save
    data = {
        "url": url,
        "timestamp": datetime.now().isoformat(),
        "result": result,
        "total_links_processed": len(fetch_website_links(url)) if result else 0
    }
    
    # Save to JSON file
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"Result saved to: {filepath}")
    return filepath

# Test the function with system and user prompts
result = select_relative_links("https://www.trendyol.com/")
print("Generated result:")
print(result)
print("\n" + "="*50 + "\n")

# Save the result to JSON file
saved_file = save_result_to_json(result, "https://www.trendyol.com/")
print(f"Saved to: {saved_file}")


Generated result:
```json
[
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-17-cep-telefonu-x-c103498-a292-v10632549"
  },
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-17-air-cep-telefonu-x-c103498-a292-v10632550"
  },
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-17-pro-cep-telefonu-x-c103498-a292-v10632551"
  },
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-17-pro-max-cep-telefonu-x-c103498-a292-v10632552"
  },
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-16-cep-telefonu-x-c103498-a292-v10577547"
  },
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-16-plus-cep-telefonu-x-c103498-a292-v10591876"
  },
  {
    "category": "electronics",
    "url": "https://www.trendyol.com/iphone-16-pro-cep-telefonu-x-c103498-a292-v10577551"
  },
    {
    "category": "electronics",
    "url": "https://www.tren

## Agentic AI logic

In [36]:
link_system_prompt = """
     You are a agentic ai your mission is to visualize the product and category list you have been given
     the format of the product link and category list is JSON you need to get it in that format and turn it into 
     .md formatted broschure to show off you shouldnt make any explainations just the .md formatted brochure you need to answer as
"""

In [37]:
def get_user_prompt_brochure(url, company_name):
     user_prompt = f"""
          you are a white collar employee working for the {company_name} company you create brochure for them
          can you create me a stylish .md brochure witht the given json formatted list of product url and product list? 
          this list should show off the product name and the category not the product url so you should be carefull with that
          from the urls extract each products name and the categories are already there
     """
     
     relative_links = select_relative_links(url);
     user_prompt += f"\n{relative_links}"
     return user_prompt;

In [38]:
def create_brochure(url):
     model_name = "gemini-2.0-flash"
     client = genai.Client(
          api_key=os.getenv("GEMINI_API_KEY")
     )
     
     # Get user prompt with links
     user_prompt = get_user_prompt_brochure("https://www.trendyol.com/", "Kem Limited LCC.");
     
     # CORRECT WAY: Combine system and user prompts into a single prompt
     combined_prompt = f"{link_system_prompt}\n\n{user_prompt}"
     
     # Create contents with the combined prompt
     contents = [
         types.Content(
             role="user",  # Only "user" and "model" roles exist in Gemini
             parts=[
                 types.Part.from_text(text=combined_prompt),
             ],
         ),
     ]
     
     # Generate content
     try:
         response = client.models.generate_content(
             model=model_name,
             contents=contents,
         )
         return response.text
     except Exception as e:
         return f"Error generating content: {str(e)}"

In [39]:
result = create_brochure("https://www.trendyol.com/");
display(Markdown(result));

```md
# KEM Limited LCC - Product Showcase

## Electronics

*   **iPhone 17:** The future in your hands.
    [Learn More](https://www.trendyol.com/iphone-17-cep-telefonu-x-c103498-a292-v10632549)

*   **iPhone 17 Air:** Lightweight and powerful.
    [Learn More](https://www.trendyol.com/iphone-17-air-cep-telefonu-x-c103498-a292-v10632550)

*   **iPhone 17 Pro:** Innovation redefined.
    [Learn More](https://www.trendyol.com/iphone-17-pro-cep-telefonu-x-c103498-a292-v10632551)

*   **iPhone 17 Pro Max:** The ultimate experience.
    [Learn More](https://www.trendyol.com/iphone-17-pro-max-cep-telefonu-x-c103498-a292-v10632552)

*   **iPhone 16:** Classic, refined, and reliable.
    [Learn More](https://www.trendyol.com/iphone-16-cep-telefonu-x-c103498-a292-v10577547)

*   **iPhone 16 Plus:** Bigger screen, bigger possibilities.
    [Learn More](https://www.trendyol.com/iphone-16-plus-cep-telefonu-x-c103498-a292-v10591876)

*   **iPhone 16 Pro:** Professional-grade performance.
    [Learn More](https://www.trendyol.com/iphone-16-pro-cep-telefonu-x-c103498-a292-v10577551)

*   **iPhone 16 Pro Max 256GB:** Power and storage in perfect harmony.
    [Learn More](https://www.trendyol.com/apple/iphone-16-pro-max-256gb-siyah-titanyum-p-857296077)

*   **iPhone 16e 128GB:** Style and efficiency combined.
    [Learn More](https://www.trendyol.com/apple/iphone-16e-128gb-siyah-p-900754126)

*   **iPhone 13:** Still a great choice!
    [Learn More](https://www.trendyol.com/iphone-13-cep-telefonu-x-c103498-a292-v613184)

*   **iPhone 13 Mini:** Compact yet mighty.
    [Learn More](https://www.trendyol.com/iphone-13-mini-cep-telefonu-x-c103498-a292-v613188)

*   **iPhone 13 Pro:** Unleash your creativity.
    [Learn More](https://www.trendyol.com/iphone-13-pro-cep-telefonu-x-c103498-a292-v613190)

*   **iPhone 13 Pro Max:** The ultimate professional tool.
    [Learn More](https://www.trendyol.com/iphone-13-pro-max-cep-telefonu-x-c103498-a292-v613194)

*   **iPhone 14:** Elevating the standard.
    [Learn More](https://www.trendyol.com/iphone-14-cep-telefonu-x-c103498-a292-v1195053)

*   **iPhone 14 Plus:** More to see, more to love.
    [Learn More](https://www.trendyol.com/iphone-14-plus-cep-telefonu-x-c103498-a292-v1195068)

*   **iPhone 14 Pro:** Experience the difference.
    [Learn More](https://www.trendyol.com/iphone-14-pro-cep-telefonu-x-c103498-a292-v1195066)

*   **iPhone 14 Pro Max:** The best just got better.
    [Learn More](https://www.trendyol.com/iphone-14-pro-max-cep-telefonu-x-c103498-a292-v1195067)

*   **iPhone 15:** A new era of iPhone.
    [Learn More](https://www.trendyol.com/iphone-15-cep-telefonu-x-c103498-a292-v1218519)

*   **iPhone 15 Plus:** Larger and more immersive.
    [Learn More](https://www.trendyol.com/iphone-15-plus-cep-telefonu-x-c103498-a292-v1218521)

*   **iPhone 15 Pro:** Redefining what's possible.
    [Learn More](https://www.trendyol.com/iphone-15-pro-cep-telefonu-x-c103498-a292-v1218520)

*   **iPhone 15 Pro Max:** The pinnacle of innovation.
    [Learn More](https://www.trendyol.com/iphone-15-pro-max-cep-telefonu-x-c103498-a292-v1218522)

*   **Cell Phones:** Explore our wide selection.
    [Learn More](https://www.trendyol.com/cep-telefonu-x-c103498)

*   **PlayStation 5:** Enter a new world of gaming.
    [Learn More](https://www.trendyol.com/playstation-5-x-c144046)

## Featured Brands

*   **Decathlon:** Your destination for sports and outdoor gear.
    [Learn More](https://www.trendyol.com/decathlon-x-b150625)

*   **IKEA:** Creating a better everyday life.
    [Learn More](https://www.trendyol.com/ikea-x-b108268)

*   **UGG:** Comfort and style redefined.
    [Learn More](https://www.trendyol.com/ugg-x-b142603)

*   **Adidas Samba:** Iconic style, timeless appeal.
    [Learn More](https://www.trendyol.com/adidas-samba-y-s10410)

*   **Bargello:** Experience exquisite fragrances.
    [Learn More](https://www.trendyol.com/bargello-x-b109919)

*   **Nike Air Force:** Elevate your sneaker game.
    [Learn More](https://www.trendyol.com/nike-air-force-y-s7252)

## Cosmetics

*   **Makeup Set:** Enhance your natural beauty.
    [Learn More](https://www.trendyol.com/makyaj-seti-x-c104019)

## Investment Gold

*   **Gram Gold:** A secure investment.
    [Learn More](https://trendyol.com/gram-altin-x-c103535)

*   **Quarter Gold:** A classic choice for investment.
    [Learn More](https://trendyol.com/ceyrek-altin-x-c103533)

* * *

**KEM Limited LCC - Your Trusted Partner.**
```